# Task 1-d — HaluEval 교차 검증

## 목적

지금까지의 판정기 비교(RAGTruth, 360개 응답 표본)는 모두 같은 표본 위에서 반복적으로
이루어졌다. 이는 지도교수가 지적한 과적합 우려의 일부로, 점수 산출 방식과 판정기 선택이
이 특정 표본에 유리하게 맞춰졌을 가능성을 배제할 수 없다. 이번 실험은 완전히 다른
데이터셋인 HaluEval에서 동일한 판정기 비교를 반복하여, RAGTruth에서 얻은 결론
(MiniCheck가 우수하다, 근거 전체를 훑는 것이 절단보다 낫다)이 데이터셋을 바꿔도
유지되는지 확인한다.

## 데이터셋 구조의 차이

RAGTruth는 응답 내 환각 위치가 문자 구간으로 표시되어 문장 단위 평가가 가능했다.
HaluEval은 질문/문서당 정상 응답(right)과 환각 응답(hallucinated)이 쌍으로만 제공되며
구간 정보가 없다. 따라서 이번 실험은 **응답 단위** 판정으로 수행한다. 이는 검증 단위
실험(Task 1의 원래 목적)의 재현이 아니라, 판정기 자체의 일반화 여부를 확인하는 것이
목적임을 분명히 한다.

## 사전 확인 사항 — 길이 편향

사전 점검에서 QA 도메인의 정상 응답은 평균 13자, 환각 응답은 평균 65자로 길이 차이가
체계적으로 나타났다. 응답 길이만으로도 어느 정도 구분이 가능하다면 판정기의 AUROC가
실제 의미 판단이 아닌 길이 편향에서 비롯되었을 수 있다. 이를 판별하기 위해 응답 길이만을
사용하는 단순 기준선을 함께 측정한다.

## 사전 정의 기준

RAGTruth에서 확정한 기준을 그대로 적용한다: AUROC 0.80 이상, 문장(응답)당 100ms 이하.
길이 기준선 대비 판정기의 AUROC 우위가 뚜렷하지 않다면 판별력을 의미 판단으로 보기 어렵다.

## 1. 환경 준비

In [ ]:
!pip -q install sentencepiece scikit-learn pandas matplotlib nltk
!git clone --depth 1 https://github.com/RUCAIBox/HaluEval.git 2>/dev/null || echo already cloned
!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" 2>&1 | tail -2

import nltk
for pkg in ['punkt', 'punkt_tab']:
    try: nltk.download(pkg, quiet=True)
    except Exception as e: print(pkg, 'download 실패:', e)

import json, time, random, gc
import numpy as np, pandas as pd, torch, transformers
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| device', DEVICE)

## 2. 데이터 적재 및 표본 구성

In [ ]:
BASE = 'HaluEval/data'

def load_pairs(path, ref_key, right_key, hallu_key, n):
    rows = []
    with open(path) as f:
        lines = f.readlines()
    random.Random(SEED).shuffle(lines)
    for line in lines[:n]:
        d = json.loads(line)
        ref = d[ref_key] if isinstance(ref_key, str) else ref_key(d)
        rows.append({'reference': ref, 'claim': d[right_key], 'y': 0})
        rows.append({'reference': ref, 'claim': d[hallu_key], 'y': 1})
    return pd.DataFrame(rows)

N_PER_DOMAIN = 200   # 도메인당 정상+환각 각 200건 = 400 예시. 시간 여유 있으면 상향

qa = load_pairs(f'{BASE}/qa_data.json', 'knowledge', 'right_answer', 'hallucinated_answer', N_PER_DOMAIN)
qa['domain'] = 'QA'

summ = load_pairs(f'{BASE}/summarization_data.json', 'document', 'right_summary', 'hallucinated_summary', N_PER_DOMAIN)
summ['domain'] = 'Summary'

dialog = load_pairs(
    f'{BASE}/dialogue_data.json',
    lambda d: d['knowledge'] + '\n' + d['dialogue_history'],
    'right_response', 'hallucinated_response', N_PER_DOMAIN)
dialog['domain'] = 'Dialogue'

df = pd.concat([qa, summ, dialog], ignore_index=True)
print(f'총 예시 {len(df):,}개 (도메인당 정상 {N_PER_DOMAIN} + 환각 {N_PER_DOMAIN})')
df.groupby('domain').y.agg(['size','mean'])

## 3. 길이 기준선 — 판별력이 길이 편향에서 오는지 확인

In [ ]:
df['claim_len'] = df.claim.str.len()

print('=== 도메인별 정상/환각 응답 길이 ===')
print(df.groupby(['domain','y']).claim_len.mean().round(1))

print('\n=== 길이만으로 판별했을 때의 AUROC (도메인별) ===')
len_auroc = {}
for d, g in df.groupby('domain'):
    a = roc_auc_score(g.y, g.claim_len)
    len_auroc[d] = a
    print(f'  {d:10} AUROC {a:.3f}  (0.5에서 먼 만큼 길이만으로 구분 가능함을 의미)')

**해석 지침** — 길이 기준선의 AUROC가 이미 0.7~0.8을 넘는 도메인이 있다면, 그 도메인에서
판정기의 AUROC가 높게 나와도 의미 판단 능력을 증명하는 것이 아닐 수 있다. 이런 도메인은
결론을 내릴 때 가중치를 낮추거나 별도로 유보한다.

## 4. MiniCheck 평가 (응답 단위)

In [ ]:
results, all_scores = [], {}

try:
    from minicheck.minicheck import MiniCheck
    mc = MiniCheck(model_name='flan-t5-large', enable_prefix_caching=False)
    docs, claims = df.reference.tolist(), df.claim.tolist()
    t0 = time.perf_counter()
    _, raw_prob, _, _ = mc.score(docs=docs, claims=claims)
    el = time.perf_counter() - t0
    s = 1.0 - np.asarray(raw_prob)
    all_scores['MiniCheck'] = s
    results.append({'judge': 'MiniCheck-FlanT5-L', 'AUROC': roc_auc_score(df.y, s),
                    'AUPRC': average_precision_score(df.y, s), 'ms_per_item': el*1000/len(df)})
    print(f"MiniCheck  AUROC {results[-1]['AUROC']:.3f}  AUPRC {results[-1]['AUPRC']:.3f}  "
          f"{results[-1]['ms_per_item']:.1f} ms/응답")
    del mc
except Exception as e:
    print(f'[MiniCheck 실패] {type(e).__name__}: {str(e)[:300]}')
finally:
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 5. DeBERTa 창 분할 평가 (RAGTruth 실험과 동일 방식)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT = 'microsoft/deberta-large-mnli'
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).to(DEVICE).eval()
ENT_IDX = [i for i, l in model.config.id2label.items() if l.lower().startswith('entail')][0]

WIN_TOKENS, STRIDE, BATCH = 400, 200, 16

def make_windows(text):
    ids = tok(text, add_special_tokens=False)['input_ids']
    if len(ids) <= WIN_TOKENS: return [text]
    out = []
    for i in range(0, len(ids), STRIDE):
        chunk = ids[i:i+WIN_TOKENS]
        if not chunk: break
        out.append(tok.decode(chunk))
        if i + WIN_TOKENS >= len(ids): break
    return out

@torch.no_grad()
def entail_prob(prem, hyp):
    enc = tok(prem, hyp, truncation=True, max_length=512, padding=True, return_tensors='pt').to(DEVICE)
    return model(**enc).logits.softmax(-1)[:, ENT_IDX].cpu().numpy()

win_cache = {i: make_windows(ref) for i, ref in enumerate(df.reference)}
print('평균 창 개수: %.2f' % np.mean([len(v) for v in win_cache.values()]))

flat_p, flat_h, owner = [], [], []
for idx, row in enumerate(df.itertuples()):
    for w in win_cache[idx]:
        flat_p.append(w); flat_h.append(row.claim); owner.append(idx)

if DEVICE == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
ent = []
for i in range(0, len(flat_p), BATCH):
    ent.extend(entail_prob(flat_p[i:i+BATCH], flat_h[i:i+BATCH]).tolist())
if DEVICE == 'cuda': torch.cuda.synchronize()
el = time.perf_counter() - t0

ent_max = np.zeros(len(df))
for o, e in zip(owner, ent):
    ent_max[o] = max(ent_max[o], e)
s_deberta = 1.0 - ent_max
all_scores['DeBERTa-window'] = s_deberta

results.append({'judge': 'DeBERTa-MNLI (창분할)', 'AUROC': roc_auc_score(df.y, s_deberta),
                'AUPRC': average_precision_score(df.y, s_deberta), 'ms_per_item': el*1000/len(df)})
print(f"DeBERTa 창분할  AUROC {results[-1]['AUROC']:.3f}  AUPRC {results[-1]['AUPRC']:.3f}  "
      f"{results[-1]['ms_per_item']:.1f} ms/응답")

del model; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

## 6. RAGTruth 결과와 비교

In [ ]:
RAGTRUTH_REF = {
    'MiniCheck-FlanT5-L':    {'AUROC': 0.814, 'AUPRC': 0.376, 'ms': 379.2},
    'DeBERTa-MNLI (창분할)': {'AUROC': 0.744, 'AUPRC': 0.224, 'ms': 388.0},
}

res = pd.DataFrame(results)
print('=== RAGTruth(문장 단위) vs HaluEval(응답 단위) ===\n')
for _, r in res.iterrows():
    ref = RAGTRUTH_REF.get(r.judge)
    print(f'{r.judge}')
    print(f'  RAGTruth : AUROC {ref["AUROC"]:.3f}  AUPRC {ref["AUPRC"]:.3f}  {ref["ms"]:.0f}ms' if ref else '  RAGTruth 결과 없음')
    print(f'  HaluEval : AUROC {r.AUROC:.3f}  AUPRC {r.AUPRC:.3f}  {r.ms_per_item:.0f}ms')
    if ref:
        diff = r.AUROC - ref['AUROC']
        note = '재현됨 (차이 작음)' if abs(diff) < 0.05 else ('HaluEval에서 더 우수' if diff > 0 else 'HaluEval에서 저하')
        print(f'  AUROC 차이: {diff:+.3f} → {note}')
    print()

## 7. 도메인별·길이 대비 성능

In [ ]:
for name, s in all_scores.items():
    print(f'=== {name} — 도메인별 AUROC (참고: 길이 기준선 AUROC) ===')
    for d, g in df.groupby('domain'):
        idx = g.index
        a = roc_auc_score(g.y, s[idx])
        print(f'  {d:10} 판정기 {a:.3f}   길이기준선 {len_auroc[d]:.3f}   '
              f'{"판정기가 길이보다 우수" if a > len_auroc[d] + 0.03 else "길이 기준선과 유사 — 해석 주의"}')
    print()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
labels = list(res.judge) + ['Length baseline (avg)']
values = list(res.AUROC) + [np.mean(list(len_auroc.values()))]
colors = ['steelblue']*len(res) + ['gray']
ax.bar(labels, values, color=colors)
ax.axhline(0.80, color='r', ls='--', label='target AUROC 0.80')
ax.axhline(0.50, color='gray', ls=':', lw=1)
ax.set_ylabel('AUROC'); ax.set_ylim(0,1); ax.legend()
ax.set_title('HaluEval cross-dataset check')
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
res.to_csv('halueval_judge_comparison.csv', index=False)
df.assign(**{n: s for n, s in all_scores.items()}).to_csv('halueval_scores.csv', index=False)
print('저장: halueval_judge_comparison.csv, halueval_scores.csv')
res